# Bailey, Borwein, Lopez de Prado and Zhu (2014): Pseudo-Mathematics and Financial Charlatanism

*Reflections on the paper and on the questions I worked through while reading it.*

## What the paper is actually about

A backtest is a historical simulation of a trading strategy: you rerun a rule against past prices to see what it would have earned. The central distinction the paper cares about is between in-sample (IS) performance, measured on the data used while designing and tuning the strategy, and out-of-sample (OOS) performance, measured on data the strategy never saw during that tuning process. A backtest is only trustworthy if IS and OOS performance roughly agree.

The paper's whole argument rests on one simple but underappreciated fact. If you try enough parameter combinations and only report the best one, you will find something that looks great IS purely by chance, even if there is no real skill anywhere in the strategy. This is called overfitting, a concept borrowed from machine learning, and it means the model is targeting the specific noise in one sample of data rather than any real, repeatable structure.

Their running toy example is "crossing moving averages": own a stock whenever its short window average price exceeds its long window average price. The window lengths are free parameters, and it is trivial to search over many combinations until one happens to match the noise in a given historical sample.

## Why the usual fixes do not work here

Two existing toolkits try to guard against overfitting: econometric corrections that adjust p-values for the number of trials attempted (White's Reality Check, Romano and Wolf's stepwise testing), and machine learning overfitting theory. Neither transfers cleanly to backtests.

The econometric tools assume you are testing a regression with an explicit forecast and a defined confidence band. Most trading strategies just emit a qualitative signal like buy or sell, with no such structure, and no stated confidence or holding horizon. The machine learning tools are usually built to catch a single overly flexible model memorizing noise inside one training run, not a researcher who tries many separate strategy configurations and reports only the one that worked. And critically, almost none of these tools track N, the number of trials attempted, which turns out to be the single most important number for judging whether a good backtest means anything.

A subtlety worth holding onto here: a guardrail like the Akaike Information Criterion is not being applied incorrectly when a researcher passes it after twenty tries. It is applied correctly, every single time. The problem is that the guardrail's stated confidence level, say 95 percent, is a per trial budget. It has no memory of how many times it has already been spent. Run a test with a 5 percent false positive rate twenty times, and the probability that at least one of those twenty runs falsely passes is

$$1 - (1 - 0.05)^{20} \approx 64\%$$

not 5 percent. This is the general phenomenon of multiple comparisons, sometimes called p-hacking or the garden of forking paths, and it is the precise gap that the paper's own tool, the Minimum Backtest Length, is built to fill.

## The Sharpe ratio and the statistics of overfitting

The Sharpe ratio is the standard way to score a strategy: the average excess return over a period, divided by the standard deviation of those same returns, annualized by the square root of the sampling frequency q:

$$SR = \frac{\mu}{\sigma}\sqrt{q}$$

Since mu and sigma are never actually known, only estimated from a finite historical sample, the estimated Sharpe ratio $\widehat{SR}$ is itself a random variable with its own sampling noise. As the number of years of data y grows, its distribution converges to

$$\widehat{SR} \xrightarrow{a} \mathcal{N}\left(SR,\ \frac{1 + SR^2/2q}{y}\right)$$

The paper's key move is to ask a sharper question. If you have N strategies, every single one with a true Sharpe ratio of exactly zero, pure noise with no skill anywhere, how impressive should the best of them look purely by chance? This is a question about the expected maximum of N random draws, answered using extreme value theory, specifically the Gumbel distribution:

$$E[\max_N] \approx (1-\gamma)Z^{-1}\left(1-\frac{1}{N}\right) + \gamma Z^{-1}\left(1-\frac{1}{N}e^{-1}\right)$$

where $\gamma \approx 0.5772$ is the Euler-Mascheroni constant, a fixed number that shows up repeatedly in extreme value statistics, and $Z^{-1}$ is the inverse of the standard normal CDF. A much simpler upper bound is $\sqrt{2\ln N}$.

The concrete number that stuck with me: with only N = 10 completely skill-less strategies, the expected Sharpe ratio of the best one is already 1.57. Not "could be as high as". Expected. That is the average outcome if you ran this exact experiment over and over, not a lucky anecdote from a single run.

## Minimum Backtest Length

This result flips into something actionable. If mu is zero and you have y years of data instead of one, you rescale the expected maximum by $y^{-1/2}$, and solving for y instead of for the expected maximum gives Theorem 2, the Minimum Backtest Length:

$$MinBTL \approx \left(\frac{(1-\gamma)Z^{-1}\left[1-\frac1N\right] + \gamma Z^{-1}\left[1-\frac1N e^{-1}\right]}{E[\max_N]}\right)^2 < \frac{2\ln N}{E[\max_N]^2}$$

Read practically: given how many independent parameter combinations you tried, N, this tells you how much backtest history you would need before a noise driven strategy could no longer fake a given Sharpe ratio. With only five years of data, you should not try more than about 45 independent configurations, or you are close to guaranteed to manufacture a strategy with an IS Sharpe ratio of 1 and a true OOS Sharpe ratio of zero. With two years of data, that ceiling drops to about seven trials.

The practical warning underneath all of this: a researcher who does not report N makes it impossible for anyone else to judge how much to trust their backtest. A Sharpe ratio by itself is not evidence of anything without knowing how hard it was searched for.

One caveat worth remembering: this assumes all N trials are independent. In practice many parameter variations are highly correlated with each other, a 20 day and a 21 day moving average window will produce nearly identical results, so the raw count of trials overstates the true number of independent shots taken. Principal Component Analysis is the natural tool for fixing this. Stack every trial's return series into a table, compute the correlation matrix between all N trials, and look at how many eigenvalues, the principal components, are needed to explain most of the variance across them. That count is the effective number of independent trials, $N_{eff}$, and it is $N_{eff}$, not the raw N, that should go into the formulas above. PCA answers "how many genuinely different bets did I actually place", which is a separate question from "how many times did I press the button".

## Model complexity

A one parameter model with two settings gives N = 2 trials. Add four more binary parameters and the count becomes $2^5 = 32$. This is why complexity is dangerous: adding parameters grows the number of trials exponentially, not linearly. Just seven independent binary parameters already gives 128 trials, enough for an expected maximum Sharpe ratio above 2.6 out of pure noise, per the same formula above.

There are really two separate ways complexity causes trouble, and it is worth keeping them apart. The first is the one the paper actually proves things about: a large hyperparameter space, more settings to search over, inflates N in exactly the sense of the expected maximum formula. The second is more familiar from machine learning generally: a single sufficiently flexible model, like a neural network with many internal weights, can memorize noise in one training run without needing any external search over separate trials at all. The paper's own tools are built for the first mechanism. The second is the more classical overfitting story, and it is part of why the introduction argues that standard machine learning overfitting theory does not transfer cleanly to backtests.

Fermi, recalling a remark of von Neumann, put the general point memorably: with four parameters I can fit an elephant, and with five I can make him wiggle his trunk.

## When overfitting is merely disappointing, and when it is actively harmful

This is the part of the paper I found the most genuinely surprising, and it directly answers a question I had at the very start about why IS and OOS should even be expected to agree in the first place.

The paper's simulations start from a pure random walk: no trend, no memory, each step drawn independently of the last. Split it into an IS half and an OOS half, and pick whichever of a thousand such walks looks best IS. The result: the selected walk's IS Sharpe ratio is inflated, clustering between 1.2 and 2.6, but its OOS Sharpe ratio stays centered around zero, the true mean of the process. Selecting for a good IS Sharpe ratio told you nothing about OOS, because nothing links the two halves together in a memoryless process. Overfitting here is disappointing, but not actively harmful.

Everything changes once the data has memory. The paper shows two ways to introduce this. The first is a global constraint: force the whole series, IS and OOS combined, to average out to a fixed value. Now whatever inflated the IS half must be paid back somewhere, and it gets paid back in the OOS half, by construction. The second, less artificial way is ordinary mean reversion, modeled as a first order autoregressive process where each value drifts back toward a long run average at a rate governed by a coefficient $\phi$:

$$m_\tau = (1-\phi)\mu + \phi m_{\tau-1} + \sigma \varepsilon_\tau$$

Both produce the same qualitative result: a strongly negative relationship between IS and OOS Sharpe ratios. The better a strategy looks IS, the worse it does OOS.

The mechanism behind the autocorrelation case is worth writing out properly, because it is not a hard accounting rule the way the global constraint is. A high IS Sharpe ratio requires the path to have climbed to an elevated level by the end of the IS period, since Sharpe ratio is essentially return over volatility, and return over a stretch of time is where the path ended up relative to where it started. OOS then starts from exactly that elevated level, because IS and OOS are one continuous path, not two separately drawn ones. Mean reversion means that wherever the path is sitting right now directly predicts where it is expected to go next: above the long run average, the pull is downward. So the very same elevated ending point that made IS look impressive becomes the starting point mean reversion works against during OOS. Formally,

$$E_{\delta T}[m_T] - m_{\delta T} = (1-\phi^T)(\mu - m_{\delta T})$$

Since $1 - \phi^T$ is always positive for $\phi \in (0,1)$, a higher ending value $m_{\delta T}$, meaning a higher IS Sharpe ratio, makes $(\mu - m_{\delta T})$ more negative, which makes the expected OOS move more negative too, and therefore the OOS Sharpe ratio lower.

I asked myself, and then out loud, whether momentum would flip this result, and it does, directionally. If the process trended rather than reverted, an elevated ending point would predict continuation rather than pullback, and the IS to OOS relationship would turn positive instead of negative. But there is a real constraint hiding here. The paper's autoregressive model requires $\phi$ strictly between negative one and one for the process to be stationary and to converge back to $\mu$ at all. A process with $\phi$ greater than or equal to one is explosive, not stationary, and simply cannot be represented in this framework. True persistent momentum is therefore not a special case the paper chose to exclude, it is structurally incompatible with the convergence result the whole proof depends on. They picked mean reversion deliberately, both because it makes the sharper and more alarming point, that overfitting can hurt you rather than just disappoint you, and because it happens to be a well documented real feature of hedge fund performance streams specifically, more so than persistent trending.

This also gave me a clean way to separate three genuinely different reasons IS and OOS performance can diverge, which I had originally lumped together before working through the paper.

1. Pure overfitting from the expected maximum result, present even in a perfectly memoryless, perfectly stationary world, arising purely from selecting the best of N noisy draws.
2. A compensation effect from memory in the data itself, either a global constraint or autocorrelation, which can turn that overfitting from merely disappointing into actively harmful.
3. Genuine non-stationarity in the real world, entirely outside anything this paper models: macro regime shifts, volatility regime changes, structural breaks, crowding and arbitrage, survivorship and look-ahead bias in the data itself.

The paper isolates the first mechanism, and partially the second, deliberately holding the third constant by simulation, precisely so it can prove that overfitting alone, with nothing else going on, is already enough to fool you.

## Is this fraud, and the practical demonstration

The paper draws a direct parallel between undisclosed backtest overfitting and a classic scam. Send half of $2^n x$ recipients a bullish forecast and half a bearish one, drop everyone who got the wrong call, and repeat n times. The x survivors have now seen n consecutive correct forecasts, which looks miraculous, and is guaranteed by construction. Not reporting N when presenting a backtest is the same trick with a different mechanism: the manager only publicizes the model that worked and says nothing about the failed attempts. They draw the same parallel to clinical drug trials that only publicize the best outcomes among many patients, which is part of why initiatives demanding full disclosure of trial results, positive and negative, exist in medicine.

They demonstrate this concretely rather than just proving it abstractly. Take a pure random walk, four trading rule parameters, entry day, holding period, stop loss, and long or short, giving 8,800 combinations to search. The winning combination achieves an annualized Sharpe ratio of 1.27, with a Probabilistic Sharpe Ratio statistic of 2.83, implying under a 1 percent probability that the true Sharpe ratio is below zero. None of it is real. The data was random by construction. Even a statistic explicitly built to be more rigorous than the plain Sharpe ratio cannot rescue you here, because it still has no way of knowing that 8,800 combinations were searched before this one was reported.

## Reflection

Coming into this paper, I understood overfitting as a somewhat abstract machine learning concern, something you guard against with train test splits and cross validation, largely solved territory. What this paper changed is realizing that in finance specifically, the standard guardrails from that world do not transfer, precisely because the object being overfit, a full strategy with an entry rule, an exit rule, a stop loss, a position size, is not the kind of thing those guardrails were built to check, and because almost nobody in finance reports the one number, N, that would let you sanity check a result at all.

The part that stuck with me hardest is the distinction between overfitting that is merely disappointing and overfitting that is actively harmful. I would have guessed, before working through this, that the worst case for an overfit strategy is that it just does nothing special going forward, regression to the mean in the loosest sense. Seeing that ordinary mean reversion, a completely mundane and common property of financial time series, converts that into a strategy that is expected to actively lose money precisely because it looked good in the past, was a genuinely new mechanism to me, not just a restatement of past performance does not guarantee future results.

The multiple comparisons framing also reorganized something for me. I used to think of a passed statistical test as a fact about the one result sitting in front of me. It is really a fact about a single trial's error budget, and that budget gets spent every time the test is run, whether or not anyone bothers to keep count. A model that passes AIC honestly, twenty times in a row of attempts, has not proven anything special about the twentieth pass in isolation, because the 95 percent confidence attached to it was only ever a promise about being run once.

The PCA point was a useful correction to my own instinct too. My first reaction to Minimum Backtest Length was to treat N as simply whatever number of parameter combinations were literally tried. Realizing that correlated trials are not independent shots at getting lucky, and that the honest N to use is the effective number of independent bets, not the raw count of button presses, made the whole framework feel less like a fixed formula and more like a question you have to actually investigate for any given search.

And the momentum question sharpened something I would not have noticed on my own: the direction of the IS to OOS relationship is not a universal law, it is entirely a property of the memory structure in the data, mean reverting data punishes overfitting, trending data would reward it, and a pure random walk is indifferent to it. The paper chose the punishing case deliberately, both because it is the more alarming and useful warning, and because it happens to be the realistic case for the kind of performance streams, hedge fund returns, that most concern them.

If I had to compress the whole paper into one sentence I would want to keep, it would be something like this. A good backtest, without a disclosed number of trials and without a disclosed backtest length relative to that number, is not evidence, it is a survivor, and a survivor alone tells you nothing about how many others did not survive.